# Baseline Data-Quality and Anomaly-Detection Pipeline

**Companion notebook for:**  
*Governance-Aware Hybrid Decision Fusion for Data Quality Anomaly Detection in Cloud Lakehouses: A Multi-Domain Evaluation*

**Author:** Ramesh Babu Kallam  
**ORCID:** 0009-0008-5220-1775

## Purpose

This notebook prepares the three evaluation datasets, applies deterministic data-quality rules, creates controlled anomaly experiments, and evaluates the Isolation Forest and Local Outlier Factor baselines used by the hybrid decision engine.

## How to Run

1. Install the dependencies listed in `requirements.txt`.
2. Download the source datasets described in `docs/data_sources.md`.
3. Place them under `data/raw/`.
4. Run all cells from top to bottom.
5. Continue with `02_Hybrid_Decision_Engine.ipynb`.

The notebook writes reusable artifacts to `data/bronze/`, `data/silver/`, `data/experiments/`, `models/`, and `results/`.

> Runtime depends on the execution environment and dataset size. Google Colab and local Jupyter environments are both supported.


## 1. Install dependencies
Colab already includes many scientific packages, but this cell installs the versions needed for Excel and Parquet support.

In [ ]:
!pip -q install openpyxl pyarrow scikit-learn joblib

## 2. Configure Repository Storage

The default configuration uses the repository root and therefore works in a local clone, Codespace, or Jupyter environment. In Google Colab, set `USE_GOOGLE_DRIVE = True` and optionally change `GOOGLE_DRIVE_PROJECT_ROOT`.


In [ ]:
from pathlib import Path
import sys

# Set to True only when running in Google Colab with project files in Drive.
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/governance-aware-hybrid-data-quality")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE=True requires a Google Colab runtime."
        ) from exc

    drive.mount("/content/drive")
    PROJECT_ROOT = GOOGLE_DRIVE_PROJECT_ROOT
else:
    # In a cloned repository, the notebook is expected under notebooks/.
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = (
        current_dir.parent
        if current_dir.name == "notebooks"
        else current_dir
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"

for directory in [
    RAW_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    EXPERIMENT_DIR,
    RESULTS_DIR,
    MODEL_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)


In [ ]:
from pathlib import Path

print("Raw directory:", RAW_DIR)
print("\nFiles found:")

for file_path in sorted(RAW_DIR.iterdir()):
    print(
        file_path.name,
        "-",
        round(file_path.stat().st_size / (1024 * 1024), 2),
        "MB"
    )

## 3. Upload or copy source files
Required files:
- `bank-full.csv` (preferred) or `bank.csv`
- `diabetic_data.csv`
- `online_retail_II.xlsx`
- Optional: `IDS_mapping.csv`

When using Drive, copy the files into `Paper2_ZTLF/data/raw/`. For a one-time upload into the Colab session, set `USE_GOOGLE_DRIVE = False` and run the upload cell below.

In [ ]:
# Optional one-time upload. Skip when files already exist in RAW_DIR.
# from google.colab import files
# uploaded = files.upload()
# for filename, content in uploaded.items():
#   (RAW_DIR / filename).write_bytes(content)

## 4. Imports and reproducibility configuration

In [ ]:
import json
import re
import time
import hashlib
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

def calculate_false_positive_rate(y_true, y_pred):
    """
    Calculate the false-positive rate:
    FPR = FP / (FP + TN)
    """
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    denominator = fp + tn
    return fp / denominator if denominator > 0 else 0.0

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
pd.set_option('display.max_columns', 100)

## 5. Data ingestion and Bronze-layer artifact creation
Source datasets are loaded, normalized, enriched with audit metadata and deterministic record identifiers, and persisted as reproducible Parquet artifacts for downstream experiments.

In [ ]:
def normalize_column_name(name: str) -> str:
    normalized = re.sub(r'[^a-zA-Z0-9]+', '_', str(name).strip().lower())
    return re.sub(r'_+', '_', normalized).strip('_')


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [normalize_column_name(c) for c in result.columns]
    return result


def add_audit_columns(df: pd.DataFrame, dataset: str, source_file: str) -> pd.DataFrame:
    result = df.copy()
    result['_source_row_number'] = np.arange(len(result), dtype='int64')
    result['_dataset'] = dataset
    result['_source_file'] = source_file
    result['_ingestion_timestamp_utc'] = datetime.now(timezone.utc).isoformat()
    result['_record_id'] = [f'{dataset}_{i:09d}' for i in range(len(result))]
    return result


def locate_finance_file() -> Path:
    preferred = RAW_DIR / 'bank-full.csv'
    fallback = RAW_DIR / 'bank.csv'
    if preferred.exists():
        return preferred
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Place bank-full.csv or bank.csv in RAW_DIR.')

finance_path = locate_finance_file()
health_path = RAW_DIR / 'diabetic_data.csv'
retail_path = RAW_DIR / 'online_retail_II.xlsx'

for required in [health_path, retail_path]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required file: {required}')

finance_df = pd.read_csv(finance_path, sep=';')
health_df = pd.read_csv(health_path, na_values=['?'])

retail_sheets = pd.read_excel(retail_path, sheet_name=None)
retail_df = pd.concat(retail_sheets.values(), ignore_index=True)

# Stabilize known mixed-type identifier and text columns before Parquet writing.
retail_string_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Customer ID",
    "Country",
]

for column in retail_string_columns:
    if column in retail_df.columns:
        retail_df[column] = retail_df[column].astype("string")

bronze = {
    'finance': add_audit_columns(normalize_columns(finance_df), 'finance', finance_path.name),
    'healthcare': add_audit_columns(normalize_columns(health_df), 'healthcare', health_path.name),
    'retail': add_audit_columns(normalize_columns(retail_df), 'retail', retail_path.name),
}

for dataset, df in bronze.items():
    output = BRONZE_DIR / f'bronze_{dataset}.parquet'
    df.to_parquet(output, index=False)
    print(dataset, df.shape, '->', output)

## 6. Profiling
Produces dataset-level metadata, missing-value counts, type information, distinct counts, and descriptive statistics.

In [ ]:
def profile_dataset(df: pd.DataFrame, dataset_name: str):
    missing = df.isna().sum().rename('missing_count').to_frame()
    missing['missing_rate'] = missing['missing_count'] / max(len(df), 1)
    missing['dtype'] = df.dtypes.astype(str)
    missing['distinct_count'] = [df[c].nunique(dropna=True) for c in df.columns]
    missing['dataset'] = dataset_name
    missing['column_name'] = missing.index
    missing = missing.reset_index(drop=True)

    numeric_summary = df.select_dtypes(include=np.number).describe().T.reset_index()
    numeric_summary = numeric_summary.rename(columns={'index': 'column_name'})
    numeric_summary['dataset'] = dataset_name
    return missing, numeric_summary

metadata_rows, missing_frames, summary_frames = [], [], []
for dataset, df in bronze.items():
    missing, summary = profile_dataset(df, dataset)
    missing_frames.append(missing)
    summary_frames.append(summary)
    metadata_rows.append({
        'dataset': dataset,
        'rows': len(df),
        'columns': len(df.columns),
        'profiled_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    })

metadata_df = pd.DataFrame(metadata_rows)
missing_profile_df = pd.concat(missing_frames, ignore_index=True)
numeric_summary_df = pd.concat(summary_frames, ignore_index=True)

metadata_df.to_csv(RESULTS_DIR / 'dataset_metadata.csv', index=False)
missing_profile_df.to_csv(RESULTS_DIR / 'missing_value_profile.csv', index=False)
numeric_summary_df.to_csv(RESULTS_DIR / 'numeric_summary.csv', index=False)

display(metadata_df)

## 7. Deterministic rule configuration
These rules define the rule-based data-quality baseline. Rule violations are retained as auditable, record-level artifacts for later comparison with AI-only and hybrid methods.


In [ ]:
RULES = pd.DataFrame([
    ['FIN_A1_001','finance','Missing age','MISSING_VALUE','age','HIGH',0.25,'QUARANTINE'],
    ['FIN_A3_001','finance','Invalid age','DOMAIN_CONSTRAINT','age','HIGH',0.25,'QUARANTINE'],
    ['FIN_A3_002','finance','Invalid balance','DOMAIN_CONSTRAINT','balance','MEDIUM',0.15,'REVIEW'],
    ['FIN_A3_003','finance','Invalid campaign count','DOMAIN_CONSTRAINT','campaign','MEDIUM',0.15,'REVIEW'],
    ['HLT_A1_001','healthcare','Missing encounter identifier','MISSING_VALUE','encounter_id','CRITICAL',0.40,'QUARANTINE'],
    ['HLT_A1_002','healthcare','Missing patient identifier','MISSING_VALUE','patient_nbr','CRITICAL',0.40,'QUARANTINE'],
    ['HLT_A3_001','healthcare','Invalid hospital stay duration','DOMAIN_CONSTRAINT','time_in_hospital','HIGH',0.25,'REVIEW'],
    ['HLT_A3_002','healthcare','Invalid inpatient visit count','DOMAIN_CONSTRAINT','number_inpatient','MEDIUM',0.15,'REVIEW'],
    ['RTL_A1_001','retail','Missing invoice','MISSING_VALUE','invoice','CRITICAL',0.40,'QUARANTINE'],
    ['RTL_A1_002','retail','Missing stock code','MISSING_VALUE','stockcode','HIGH',0.25,'QUARANTINE'],
    ['RTL_A3_001','retail','Zero quantity','DOMAIN_CONSTRAINT','quantity','HIGH',0.25,'REVIEW'],
    ['RTL_A3_002','retail','Invalid unit price','DOMAIN_CONSTRAINT','price','HIGH',0.25,'REVIEW'],
], columns=['rule_id','dataset','rule_name','rule_type','column_name','severity','trust_penalty','recommendation'])

RULES['rule_version'] = '1.0'
RULES['enabled'] = True
RULES.to_csv(RESULTS_DIR / 'dq_rule_config.csv', index=False)
RULES

In [ ]:
def evaluate_rules(df: pd.DataFrame, dataset: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    masks = {}
    if dataset == 'finance':
        masks = {
            'FIN_A1_001': df['age'].isna(),
            'FIN_A3_001': df['age'].notna() & ~df['age'].between(18, 100),
            'FIN_A3_002': df['balance'].notna() & (df['balance'] < 0),
            'FIN_A3_003': df['campaign'].notna() & (df['campaign'] < 0),
        }
        duplicate_keys = ['_record_id']  # Source has no stable customer identifier.
    elif dataset == 'healthcare':
        masks = {
            'HLT_A1_001': df['encounter_id'].isna(),
            'HLT_A1_002': df['patient_nbr'].isna(),
            'HLT_A3_001': df['time_in_hospital'].notna() & ~df['time_in_hospital'].between(1, 14),
            'HLT_A3_002': df['number_inpatient'].notna() & (df['number_inpatient'] < 0),
        }
        duplicate_keys = ['encounter_id']
    elif dataset == 'retail':
        invoice_missing = df['invoice'].isna() | df['invoice'].astype('string').str.strip().eq('')
        stock_missing = df['stockcode'].isna() | df['stockcode'].astype('string').str.strip().eq('')
        masks = {
            'RTL_A1_001': invoice_missing.fillna(True),
            'RTL_A1_002': stock_missing.fillna(True),
            'RTL_A3_001': df['quantity'].eq(0),
            'RTL_A3_002': df['price'].isna() | (df['price'] <= 0),
        }
        duplicate_keys = ['invoice','stockcode','quantity','invoicedate','price','customer_id']
    else:
        raise ValueError(dataset)

    violation_rows = []
    record = df[['_record_id']].copy()
    record['dataset'] = dataset
    record['failed_rule_count'] = 0
    record['trust_score'] = 1.0
    record['failed_rules'] = [[] for _ in range(len(record))]

    rule_lookup = RULES.set_index('rule_id')
    for rule_id, mask in masks.items():
        mask = mask.fillna(False).astype(bool)
        meta = rule_lookup.loc[rule_id]
        record.loc[mask, 'failed_rule_count'] += 1
        record.loc[mask, 'trust_score'] -= float(meta['trust_penalty'])
        record.loc[mask, 'failed_rules'] = record.loc[mask, 'failed_rules'].map(lambda x: x + [rule_id])
        failing = df.loc[mask, ['_record_id']].copy()
        failing['dataset'] = dataset
        failing['rule_id'] = rule_id
        failing['rule_name'] = meta['rule_name']
        failing['severity'] = meta['severity']
        failing['recommendation'] = meta['recommendation']
        violation_rows.append(failing)

    duplicate_mask = df.duplicated(subset=duplicate_keys, keep=False)
    if duplicate_mask.any():
        dup = df.loc[duplicate_mask, ['_record_id']].copy()
        dup['dataset'] = dataset
        dup['rule_id'] = f'{dataset.upper()}_DUP_001'
        dup['rule_name'] = 'Duplicate business key'
        dup['severity'] = 'HIGH'
        dup['recommendation'] = 'QUARANTINE'
        violation_rows.append(dup)
        record.loc[duplicate_mask, 'failed_rule_count'] += 1
        record.loc[duplicate_mask, 'trust_score'] -= 0.25
        record.loc[duplicate_mask, 'failed_rules'] = record.loc[duplicate_mask, 'failed_rules'].map(lambda x: x + [f'{dataset.upper()}_DUP_001'])

    record['trust_score'] = record['trust_score'].clip(0, 1)
    record['rule_status'] = np.where(record['failed_rule_count'].eq(0), 'PASS', 'FAIL')
    violations = pd.concat(violation_rows, ignore_index=True) if violation_rows else pd.DataFrame()
    return record, violations

all_record_results, all_violations = [], []
for dataset, df in bronze.items():
    results, violations = evaluate_rules(df, dataset)
    all_record_results.append(results)
    if not violations.empty:
        all_violations.append(violations)

rule_record_results = pd.concat(all_record_results, ignore_index=True)
rule_violations = pd.concat(all_violations, ignore_index=True) if all_violations else pd.DataFrame()
rule_record_results.to_parquet(RESULTS_DIR / 'rule_record_results.parquet', index=False)
rule_violations.to_parquet(RESULTS_DIR / 'rule_violations.parquet', index=False)

display(rule_record_results.groupby(['dataset','rule_status']).size().rename('records').reset_index())

## 8. Controlled anomaly injection
This baseline creates reproducible experiments at 1%, 5%, and 10% anomaly rates. It preserves clean reference data and writes explicit ground-truth labels.

Implemented baseline anomaly classes:
- A1: missingness
- A2: duplicates
- A3: domain violations
- A4: statistical outliers
- A5: contextual anomalies
- A6: distribution drift
- A7: cross-attribute inconsistencies

In [ ]:
ANOMALY_RATES = [0.01, 0.05, 0.10]
ANOMALY_TYPES = ['A1','A2','A3','A4','A5','A6','A7']

DATASET_CONFIG = {
    'finance': {
        'record_key': '_record_id', 'missing_column': 'age', 'numeric_column': 'balance',
        'domain_column': 'campaign', 'context_columns': ('age','job'), 'cross_columns': ('housing','loan')
    },
    'healthcare': {
        'record_key': '_record_id', 'missing_column': 'encounter_id', 'numeric_column': 'num_lab_procedures',
        'domain_column': 'time_in_hospital', 'context_columns': ('age','time_in_hospital'), 'cross_columns': ('diabetesmed','change')
    },
    'retail': {
        'record_key': '_record_id', 'missing_column': 'invoice', 'numeric_column': 'quantity',
        'domain_column': 'price', 'context_columns': ('quantity','price'), 'cross_columns': ('quantity','price')
    },
}


def inject_anomaly(df: pd.DataFrame, dataset: str, anomaly_type: str, rate: float, seed: int):
    cfg = DATASET_CONFIG[dataset]
    out = df.copy(deep=True)
    n = max(1, int(round(len(out) * rate)))
    selected = out.sample(n=min(n, len(out)), random_state=seed).index
    original_ids = out.loc[selected, '_record_id'].astype(str).tolist()

    if anomaly_type == 'A1':
        out.loc[selected, cfg['missing_column']] = np.nan
    elif anomaly_type == 'A2':
        duplicates = out.loc[selected].copy()
        duplicates['_record_id'] = [f'{rid}_dup_{seed}' for rid in original_ids]
        out = pd.concat([out, duplicates], ignore_index=True)
        original_ids = duplicates['_record_id'].astype(str).tolist()
    elif anomaly_type == 'A3':
        col = cfg['domain_column']
        out.loc[selected, col] = -999
    elif anomaly_type == 'A4':
        col = cfg['numeric_column']
        numeric = pd.to_numeric(out[col], errors='coerce')
        scale = numeric.std(skipna=True) or 1.0
        center = numeric.median(skipna=True) or 0.0
        out.loc[selected, col] = center + 12 * scale
    elif anomaly_type == 'A5':
        c1, c2 = cfg['context_columns']
        if dataset == 'finance':
            out.loc[selected, c1] = 18
            out.loc[selected, c2] = 'retired'
        elif dataset == 'healthcare':
            out.loc[selected, c1] = '[0-10)'
            out.loc[selected, c2] = 14
        else:
            out.loc[selected, c1] = 1
            out.loc[selected, c2] = pd.to_numeric(out[c2], errors='coerce').median() * 50
    elif anomaly_type == 'A6':
        col = cfg['numeric_column']
        out.loc[selected, col] = pd.to_numeric(out.loc[selected, col], errors='coerce') * 3 + 100
    elif anomaly_type == 'A7':
        c1, c2 = cfg['cross_columns']
        if dataset == 'finance':
            out.loc[selected, [c1,c2]] = 'yes'
        elif dataset == 'healthcare':
            out.loc[selected, c1] = 'No'
            out.loc[selected, c2] = 'Ch'
        else:
            out.loc[selected, c1] = -abs(pd.to_numeric(out.loc[selected, c1], errors='coerce').fillna(1))
            out.loc[selected, c2] = abs(pd.to_numeric(out.loc[selected, c2], errors='coerce').fillna(1))
    else:
        raise ValueError(anomaly_type)

    experiment_id = f'{dataset}_{anomaly_type}_r{int(rate*100):02d}_s{seed}'
    ground_truth = pd.DataFrame({
        'experiment_id': experiment_id,
        'dataset': dataset,
        'anomaly_type': anomaly_type,
        'anomaly_rate': rate,
        'record_id': original_ids,
        'ground_truth_label': 1,
        'injection_seed': seed,
    })
    out['_experiment_id'] = experiment_id
    return experiment_id, out, ground_truth

experiment_registry = []
ground_truth_frames = []

for d_idx, (dataset, reference_df) in enumerate(bronze.items()):
    for a_idx, anomaly_type in enumerate(ANOMALY_TYPES):
        for r_idx, rate in enumerate(ANOMALY_RATES):
            seed = RANDOM_SEED + d_idx*1000 + a_idx*100 + r_idx
            experiment_id, corrupted, gt = inject_anomaly(reference_df, dataset, anomaly_type, rate, seed)
            corrupted.to_parquet(EXPERIMENT_DIR / f'{experiment_id}.parquet', index=False)
            ground_truth_frames.append(gt)
            experiment_registry.append({
                'experiment_id': experiment_id, 'dataset': dataset,
                'anomaly_type': anomaly_type, 'anomaly_rate': rate,
                'reference_rows': len(reference_df), 'experiment_rows': len(corrupted),
                'injected_records': len(gt), 'seed': seed,
            })

experiment_registry_df = pd.DataFrame(experiment_registry)
anomaly_ground_truth_df = pd.concat(ground_truth_frames, ignore_index=True)
experiment_registry_df.to_csv(RESULTS_DIR / 'experiment_registry.csv', index=False)
anomaly_ground_truth_df.to_parquet(RESULTS_DIR / 'anomaly_ground_truth.parquet', index=False)
display(experiment_registry_df.head())

## 9. AI-only baseline — Isolation Forest and Local Outlier Factor
Models are trained on clean reference data and scored on each corrupted experiment. High-cardinality categorical fields and technical columns are excluded to keep Colab memory bounded.

In [ ]:
TECHNICAL_COLUMNS = {c for c in next(iter(bronze.values())).columns if c.startswith('_')}
MAX_CATEGORICAL_CARDINALITY = 100
MAX_TRAIN_ROWS = {'finance': 40000, 'healthcare': 40000, 'retail': 50000}
MAX_SCORE_ROWS = {'finance': None, 'healthcare': 60000, 'retail': 80000}


def select_model_columns(df: pd.DataFrame):
    candidates = [c for c in df.columns if not c.startswith('_')]
    numeric = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]
    categorical = [c for c in candidates if c not in numeric and df[c].nunique(dropna=True) <= MAX_CATEGORICAL_CARDINALITY]
    return numeric, categorical


def build_preprocessor(numeric_columns, categorical_columns):
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipe, numeric_columns),
        ('categorical', categorical_pipe, categorical_columns),
    ], remainder='drop')

model_bundle = {}
for dataset, reference_df in bronze.items():
    numeric_cols, categorical_cols = select_model_columns(reference_df)
    feature_cols = numeric_cols + categorical_cols
    train_df = reference_df[feature_cols]
    limit = MAX_TRAIN_ROWS[dataset]
    if len(train_df) > limit:
        train_df = train_df.sample(limit, random_state=RANDOM_SEED)

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    started = time.perf_counter()
    X_train = preprocessor.fit_transform(train_df)
    preprocessing_seconds = time.perf_counter() - started

    iforest = IsolationForest(
        n_estimators=200, contamination='auto', random_state=RANDOM_SEED, n_jobs=-1
    ).fit(X_train)

    n_neighbors = min(35, max(5, len(train_df)-1))
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors, novelty=True, contamination='auto', n_jobs=-1
    ).fit(X_train)

    bundle = {
        'preprocessor': preprocessor, 'iforest': iforest, 'lof': lof,
        'numeric_columns': numeric_cols, 'categorical_columns': categorical_cols,
        'training_rows': len(train_df), 'transformed_features': X_train.shape[1],
        'preprocessing_seconds': preprocessing_seconds,
    }
    model_bundle[dataset] = bundle
    joblib.dump(bundle, MODEL_DIR / f'{dataset}_baseline_models.joblib')
    print(dataset, len(train_df), X_train.shape[1])

In [ ]:
import gc
import time
from pathlib import Path

import pandas as pd


# -------------------------------------------------------------------
# Strict Colab-safe scoring limits
# -------------------------------------------------------------------
# These limits apply to the number of source records scored per
# experiment. Because two detectors are used, the final output rows
# will be approximately twice these limits.
MAX_SCORE_ROWS = {
    "finance": 10_000,
    "healthcare": 5_000,
    "retail": 6_000,
}


# -------------------------------------------------------------------
# Output directories
# -------------------------------------------------------------------
AI_RESULT_DIR = RESULTS_DIR / "ai_results_by_experiment"
AI_RESULT_DIR.mkdir(parents=True, exist_ok=True)

COMPLETED_LOG_PATH = RESULTS_DIR / "ai_scoring_completed.csv"
FAILED_LOG_PATH = RESULTS_DIR / "ai_scoring_failed.csv"


# -------------------------------------------------------------------
# Optional cleanup
# -------------------------------------------------------------------
# Delete previously generated retail result files because they were
# created with the earlier oversized sampling logic.
deleted_retail_files = 0

for path in AI_RESULT_DIR.glob("retail_*_ai_results.parquet"):
    path.unlink()
    deleted_retail_files += 1

print(
    f"Deleted {deleted_retail_files} previous retail result files."
)


# -------------------------------------------------------------------
# Experiment scoring function
# -------------------------------------------------------------------
def score_experiment(experiment_row: pd.Series) -> pd.DataFrame:
    """
    Score one anomaly-injection experiment using Isolation Forest
    and Local Outlier Factor.

    A strict record limit is enforced for each dataset. The sampled
    records preserve the experiment's approximate anomaly proportion.
    """
    dataset = experiment_row["dataset"]
    experiment_id = experiment_row["experiment_id"]

    if dataset not in model_bundle:
        raise KeyError(
            f"No trained model bundle found for dataset: {dataset}"
        )

    bundle = model_bundle[dataset]

    experiment_path = EXPERIMENT_DIR / f"{experiment_id}.parquet"

    if not experiment_path.exists():
        raise FileNotFoundError(
            f"Experiment file not found: {experiment_path}"
        )

    df = pd.read_parquet(experiment_path)

    if "_record_id" not in df.columns:
        raise KeyError(
            f"_record_id is missing from experiment {experiment_id}"
        )

    limit = MAX_SCORE_ROWS.get(dataset)

    # ---------------------------------------------------------------
    # Strict fixed-size reproducible sampling
    # ---------------------------------------------------------------
    if limit and len(df) > limit:
        experiment_truth = anomaly_ground_truth_df.loc[
            anomaly_ground_truth_df["experiment_id"].eq(experiment_id)
        ].copy()

        injected_ids = set(
            experiment_truth["record_id"]
            .astype(str)
            .tolist()
        )

        record_ids = df["_record_id"].astype(str)
        injected_mask = record_ids.isin(injected_ids)

        injected_df = df.loc[injected_mask].copy()
        clean_df = df.loc[~injected_mask].copy()

        observed_anomaly_rate = (
            len(injected_df) / max(len(df), 1)
        )

        injected_n = min(
            len(injected_df),
            max(
                1,
                round(limit * observed_anomaly_rate)
            )
        )

        clean_n = min(
            len(clean_df),
            limit - injected_n
        )

        if injected_n > 0:
            injected_df = injected_df.sample(
                n=injected_n,
                random_state=RANDOM_SEED
            )
        else:
            injected_df = injected_df.iloc[0:0].copy()

        if clean_n > 0:
            clean_df = clean_df.sample(
                n=clean_n,
                random_state=RANDOM_SEED
            )
        else:
            clean_df = clean_df.iloc[0:0].copy()

        df = pd.concat(
            [injected_df, clean_df],
            ignore_index=True
        )

        del injected_df
        del clean_df
        gc.collect()

    # ---------------------------------------------------------------
    # Validate model feature columns
    # ---------------------------------------------------------------
    numeric_columns = bundle["numeric_columns"]
    categorical_columns = bundle["categorical_columns"]

    feature_cols = (
        numeric_columns
        + categorical_columns
    )

    missing_features = [
        column
        for column in feature_cols
        if column not in df.columns
    ]

    if missing_features:
        raise KeyError(
            f"Missing model features in {experiment_id}: "
            f"{missing_features}"
        )

    # ---------------------------------------------------------------
    # Preprocess records
    # ---------------------------------------------------------------
    preprocessing_started = time.perf_counter()

    X = bundle["preprocessor"].transform(
        df[feature_cols]
    )

    preprocessing_runtime = (
        time.perf_counter()
        - preprocessing_started
    )

    outputs = []

    detectors = [
        (
            "ISOLATION_FOREST",
            bundle["iforest"]
        ),
        (
            "LOCAL_OUTLIER_FACTOR",
            bundle["lof"]
        ),
    ]

    # ---------------------------------------------------------------
    # Score with both detectors
    # ---------------------------------------------------------------
    for detector_name, model in detectors:
        scoring_started = time.perf_counter()

        raw_predictions = model.predict(X)
        decision_scores = model.decision_function(X)

        scoring_runtime = (
            time.perf_counter()
            - scoring_started
        )

        detector_result = pd.DataFrame({
            "experiment_id": experiment_id,
            "dataset": dataset,
            "anomaly_type": experiment_row["anomaly_type"],
            "anomaly_rate": experiment_row["anomaly_rate"],
            "record_id": df["_record_id"].astype(str).values,
            "detector": detector_name,
            "anomaly_score": -decision_scores.astype(float),
            "anomaly_prediction": (
                raw_predictions == -1
            ).astype(int),
            "records_scored": len(df),
            "preprocessing_runtime_seconds": (
                preprocessing_runtime
            ),
            "batch_runtime_seconds": scoring_runtime,
        })

        outputs.append(detector_result)

    result_df = pd.concat(
        outputs,
        ignore_index=True
    )

    del df
    del X
    del outputs
    gc.collect()

    return result_df


# -------------------------------------------------------------------
# Clear objects from interrupted executions
# -------------------------------------------------------------------
for object_name in [
    "ai_result_frames",
    "ai_results_df",
    "result_df",
]:
    if object_name in globals():
        del globals()[object_name]

gc.collect()


# -------------------------------------------------------------------
# Score experiments incrementally
# -------------------------------------------------------------------
completed_records = []
failed_records = []

total_experiments = len(experiment_registry_df)

print(f"Total experiments to score: {total_experiments}")
print(f"Output directory: {AI_RESULT_DIR}")
print(f"Scoring limits: {MAX_SCORE_ROWS}")
print("-" * 90)

overall_started = time.perf_counter()

for position, (_, experiment_row) in enumerate(
    experiment_registry_df.iterrows(),
    start=1
):
    experiment_id = experiment_row["experiment_id"]
    dataset = experiment_row["dataset"]
    anomaly_type = experiment_row["anomaly_type"]
    anomaly_rate = experiment_row["anomaly_rate"]

    output_path = (
        AI_RESULT_DIR
        / f"{experiment_id}_ai_results.parquet"
    )

    experiment_started = time.perf_counter()

    try:
        # Keep correctly completed finance and healthcare files.
        # Previously generated retail files were deleted above.
        if output_path.exists():
            existing_rows = len(
                pd.read_parquet(
                    output_path,
                    columns=["record_id"]
                )
            )

            completed_records.append({
                "experiment_id": experiment_id,
                "dataset": dataset,
                "anomaly_type": anomaly_type,
                "anomaly_rate": anomaly_rate,
                "rows_written": existing_rows,
                "status": "SKIPPED_EXISTING",
                "runtime_seconds": 0.0,
                "output_path": str(output_path),
            })

            print(
                f"[{position}/{total_experiments}] "
                f"Skipped existing: {experiment_id}"
            )

            continue

        result_df = score_experiment(
            experiment_row
        )

        result_df.to_parquet(
            output_path,
            index=False
        )

        experiment_runtime = (
            time.perf_counter()
            - experiment_started
        )

        completed_records.append({
            "experiment_id": experiment_id,
            "dataset": dataset,
            "anomaly_type": anomaly_type,
            "anomaly_rate": anomaly_rate,
            "rows_written": len(result_df),
            "status": "COMPLETED",
            "runtime_seconds": experiment_runtime,
            "output_path": str(output_path),
        })

        print(
            f"[{position}/{total_experiments}] "
            f"Completed: {experiment_id} | "
            f"dataset={dataset} | "
            f"rows={len(result_df):,} | "
            f"time={experiment_runtime:.2f}s"
        )

        del result_df
        gc.collect()

    except Exception as exc:
        experiment_runtime = (
            time.perf_counter()
            - experiment_started
        )

        failed_records.append({
            "experiment_id": experiment_id,
            "dataset": dataset,
            "anomaly_type": anomaly_type,
            "anomaly_rate": anomaly_rate,
            "status": "FAILED",
            "runtime_seconds": experiment_runtime,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

        print(
            f"[{position}/{total_experiments}] "
            f"FAILED: {experiment_id} | "
            f"{type(exc).__name__}: {exc}"
        )

        gc.collect()


# -------------------------------------------------------------------
# Save execution logs
# -------------------------------------------------------------------
completed_df = pd.DataFrame(
    completed_records
)

failed_df = pd.DataFrame(
    failed_records
)

completed_df.to_csv(
    COMPLETED_LOG_PATH,
    index=False
)

failed_df.to_csv(
    FAILED_LOG_PATH,
    index=False
)

overall_runtime = (
    time.perf_counter()
    - overall_started
)

print("-" * 90)
print(
    f"Completed or skipped: {len(completed_df)}"
)
print(
    f"Failed: {len(failed_df)}"
)
print(
    f"Total runtime: "
    f"{overall_runtime / 60:.2f} minutes"
)
print(
    f"Completed log: {COMPLETED_LOG_PATH}"
)
print(
    f"Failed log: {FAILED_LOG_PATH}"
)


# -------------------------------------------------------------------
# Validate output row counts
# -------------------------------------------------------------------
expected_output_rows = {
    "finance": MAX_SCORE_ROWS["finance"] * 2,
    "healthcare": MAX_SCORE_ROWS["healthcare"] * 2,
    "retail": MAX_SCORE_ROWS["retail"] * 2,
}

print("\nExpected output rows per completed experiment:")
for dataset, expected_rows in expected_output_rows.items():
    print(
        f"{dataset}: approximately "
        f"{expected_rows:,} rows"
    )


# -------------------------------------------------------------------
# Preview execution status
# -------------------------------------------------------------------
display(completed_df.head(10))

if not failed_df.empty:
    print("\nFailed experiments:")
    display(failed_df)
else:
    print(
        "\nAll experiments completed successfully."
    )

## 10. AI-only evaluation
Metrics are calculated against the explicit injected-anomaly ground truth. Macro summaries across anomaly types and datasets should be used in the paper rather than relying on a single experiment.

In [ ]:
# Reload the incrementally saved results into a single DataFrame
ai_result_files = list(AI_RESULT_DIR.glob('*_ai_results.parquet'))
ai_results_df = pd.concat([pd.read_parquet(f) for f in ai_result_files], ignore_index=True)

gt_keys = anomaly_ground_truth_df[['experiment_id','record_id','ground_truth_label']].copy()
evaluation = ai_results_df.merge(gt_keys, on=['experiment_id','record_id'], how='left')
evaluation['ground_truth_label'] = evaluation['ground_truth_label'].fillna(0).astype(int)

metric_rows = []
for keys, group in evaluation.groupby(['experiment_id','dataset','anomaly_type','anomaly_rate','detector']):
    y_true = group['ground_truth_label'].to_numpy()
    y_pred = group['anomaly_prediction'].to_numpy()
    scores = group['anomaly_score'].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    metric_rows.append({
        'experiment_id': keys[0], 'dataset': keys[1], 'anomaly_type': keys[2],
        'anomaly_rate': keys[3], 'detector': keys[4],
        'true_positive': tp, 'false_positive': fp, 'false_negative': fn, 'true_negative': tn,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'false_positive_rate': fp / max(fp + tn, 1),
        'false_negative_rate': fn / max(fn + tp, 1),
        'pr_auc': average_precision_score(y_true, scores) if y_true.sum() > 0 else np.nan,
        'scored_rows': len(group),
        'runtime_seconds': group['batch_runtime_seconds'].iloc[0],
    })

ai_metrics_df = pd.DataFrame(metric_rows)
ai_metrics_df.to_csv(RESULTS_DIR / 'ai_evaluation_metrics.csv', index=False)

macro_summary = ai_metrics_df.groupby(['dataset','anomaly_type','detector'], as_index=False).agg(
    macro_precision=('precision','mean'), macro_recall=('recall','mean'),
    macro_f1=('f1','mean'), mean_fpr=('false_positive_rate','mean'),
    mean_fnr=('false_negative_rate','mean'), mean_pr_auc=('pr_auc','mean'),
    mean_runtime_seconds=('runtime_seconds','mean')
)
macro_summary.to_csv(RESULTS_DIR / 'ai_macro_summary.csv', index=False)
display(macro_summary.sort_values(['dataset','anomaly_type','detector']))

## 11. Baseline completion checks
Do not start the hybrid/XAI claims until every check passes and the output files are preserved.

In [ ]:
checks = {
    'three_datasets_loaded': len(bronze) == 3,
    'rule_results_created': len(rule_record_results) > 0,
    'all_63_experiments_created': len(experiment_registry_df) == 3 * 7 * 3,
    'ground_truth_created': len(anomaly_ground_truth_df) > 0,
    'two_ai_detectors_scored': ai_results_df['detector'].nunique() == 2,
    'evaluation_metrics_created': len(ai_metrics_df) > 0,
}
checks_df = pd.DataFrame({'check': checks.keys(), 'passed': checks.values()})
display(checks_df)
assert checks_df['passed'].all(), 'One or more baseline checks failed.'
print('Baseline workflow completed successfully.')

## 12. Required next prototype modules
After the baseline executes reliably, add these as separate notebooks or sections:

1. **Hybrid decision engine:** combine rule failures, normalized AI score, severity, and trust penalty into `ACCEPT`, `REPAIR`, `QUARANTINE`, or `ESCALATE`.
2. **Hybrid ablation:** compare rule-only, AI-only, hybrid without explanations, and full hybrid.
3. **Explainability:** generate record- and feature-level explanations; evaluate fidelity, stability, sparsity, and review usefulness.
4. **Repeated runs and statistics:** use multiple seeds, confidence intervals, paired tests, and effect sizes.
5. **Audit artifact:** persist policy version, model version, preprocessing version, record decision, explanation, timestamp, and lineage fields.
6. **Operational metrics:** latency, throughput, memory, and a clearly defined cost proxy.

These modules are necessary to answer the proposed research questions and distinguish Paper 2 from Paper 1.

## 13. Completion Summary

This cell reports the principal artifact locations created by the baseline pipeline. The generated files are consumed by the hybrid decision notebook.

In [ ]:
artifact_locations = {
    "project_root": PROJECT_ROOT,
    "bronze_data": BRONZE_DIR,
    "silver_data": SILVER_DIR,
    "experiment_data": EXPERIMENT_DIR,
    "models": MODEL_DIR,
    "results": RESULTS_DIR,
}

print(f"Configured experiments: {len(experiment_registry_df):,}")
for name, path in artifact_locations.items():
    print(f"{name}: {path}")